In [34]:
#Library
import pandas as pd
import numpy as np
import os

In [36]:
# ============================================================
# SETTINGS - change these to match what you want to process
# ============================================================
INPUT_FILE = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/DoE CAMS Air Qualtiy Data.xlsx"
SHEET_NAME = 'BARC'                 # which station (sheet) to convert
 # name of the file this script creates
#OUTPUT_FILE = 'BARC_daily.csv'     
OUTPUT_DIR = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/CAMS_CleanDataset"

# Automatically create filename from sheet name
OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{SHEET_NAME}_daily.csv"
)

MIN_HOURS_REQUIRED = 18             # a day needs at least this many hourly
                                     # readings to be trusted (out of 24)


In [22]:
# ============================================================
# STEP 1: Load the data
# ============================================================
# The real column headers are on the SECOND row of each sheet (row index 1),
# and there's a row of units right after it that we don't need, so we skip it.
print(f"Loading sheet '{SHEET_NAME}'...")
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

print(f"Loaded {len(df)} hourly rows.")
print("Columns found:", list(df.columns))

Loading sheet 'BARC'...
Loaded 80353 hourly rows.
Columns found: ['Date', 'Time', 'SO2', 'NO', 'NO2', 'NOX', 'CO', 'Co 8hr', 'O3', 'O38hr', 'PM2.5', 'PM10', 'Wind Speed', 'Wind Dir', 'Temperature', 'RH', 'Solar Rad', 'BP', 'Rain', 'V Wind Speed']


In [23]:
#REad BARC sheet 
cams = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
    header=0
)

print(cams.head())
print(cams.columns.tolist())

                  Date   Time  SO2   NO  NO2  NOX   CO  Co 8hr   O3  O38hr  \
0                  NaN    NaN  ppb  ppb  ppb  ppb  ppm     NaN  ppb    NaN   
1  2012-11-01 00:00:00  01:00  NaN  NaN  NaN  NaN  NaN     NaN  NaN    NaN   
2  2012-11-01 00:00:00  02:00  NaN  NaN  NaN  NaN  NaN     NaN  NaN    NaN   
3  2012-11-01 00:00:00  03:00  NaN  NaN  NaN  NaN  NaN     NaN  NaN    NaN   
4  2012-11-01 00:00:00  04:00  NaN  NaN  NaN  NaN  NaN     NaN  NaN    NaN   

   PM2.5   PM10 Wind Speed Wind Dir Temperature   RH Solar Rad   BP Rain  \
0  ug/m3  ug/m3        m/s      Deg          C°    %      w/m2   mb   mm   
1    NaN    NaN        NaN      NaN         NaN  NaN       NaN  NaN  NaN   
2    NaN    NaN        NaN      NaN         NaN  NaN       NaN  NaN  NaN   
3    NaN    NaN        NaN      NaN         NaN  NaN       NaN  NaN  NaN   
4    NaN    NaN        NaN      NaN         NaN  NaN       NaN  NaN  NaN   

  V Wind Speed  
0          m/s  
1          NaN  
2          NaN  
3     

In [24]:
#MinMaxFor all Columns
for col in cams.columns:
    values = pd.to_numeric(cams[col], errors="coerce")
    
    if values.notna().any():
        print(f"{col}: Min = {values.min()}, Max = {values.max()}")

SO2: Min = 0.01, Max = 270.92
NO: Min = 0.04, Max = 590.31
NO2: Min = 0.05, Max = 386.52
NOX: Min = 0.05, Max = 653.67
CO: Min = 0.01, Max = 27.16
Co 8hr: Min = 0.01, Max = 19.86
O3: Min = 0.04, Max = 100.86
O38hr: Min = 0.04, Max = 98.43142857142858
PM2.5: Min = 0.01, Max = 593.4
PM10: Min = 0.0, Max = 998.54
Wind Speed: Min = 0.0, Max = 52.42
Wind Dir: Min = 0.0, Max = 359.99
Temperature: Min = 0.04, Max = 44.92
RH: Min = 3.56, Max = 98.71
Solar Rad: Min = 0.08, Max = 1022.18
BP: Min = 0.15, Max = 1123.7
Rain: Min = 0.01, Max = 80.9
V Wind Speed: Min = 0.0, Max = 2.01


In [4]:
# ============================================================
# STEP 2: Clean the data
# ============================================================
# Real-world data often has small mistakes. In this file we found two:
#   (a) Some numbers were typed with a COMMA instead of a period,
#       e.g. "13,62" instead of 13.62 - Excel/pandas reads this as TEXT,
#       not as a number, so it would be ignored when we average.
#   (b) A few sensor readings are clearly wrong (negative, or absurdly
#       high, like 5000 ug/m3) - these are almost certainly equipment
#       errors, so we treat them as missing rather than let them
#       distort the daily average.

def clean_number_column(column):
    """Turn a messy column of text/numbers into a clean numeric column."""
    # First, force everything into text so we can fix the comma issue
    column = column.astype(str).str.strip()
    column = column.str.replace(',', '.', regex=False)

    # Now convert to actual numbers. Anything that still doesn't look
    # like a number (blanks, stray characters) becomes NaN ("missing").
    column = pd.to_numeric(column, errors='coerce')

    # Remove physically impossible values (sensor glitches)
    column[(column < 0) | (column > 1000)] = np.nan

    return column

df['PM2.5'] = clean_number_column(df['PM2.5'])
df['PM10'] = clean_number_column(df['PM10'])

In [25]:
# ============================================================
# STEP 2: Clean the data
# ============================================================
# Real-world data often has small mistakes. In this file we found two:
#   (a) Some numbers were typed with a COMMA instead of a period,
#       e.g. "13,62" instead of 13.62 - Excel/pandas reads this as TEXT,
#       not as a number, so it would be ignored when we average.
#   (b) A few sensor readings are clearly wrong (negative, or absurdly
#       high, like 5000 ug/m3) - these are almost certainly equipment
#       errors, so we treat them as missing rather than let them
# 

def clean_number_column(column, min_valid, max_valid):
    """
    Turn a messy column of text/numbers into a clean numeric column,
    and remove any reading outside the realistic [min_valid, max_valid]
    range for that particular parameter (sensor glitches).
    """
    # First, force everything into text so we can fix the comma issue
    column = column.astype(str).str.strip()
    column = column.str.replace(',', '.', regex=False)

    # Now convert to actual numbers. Anything that still doesn't look
    # like a number (blanks, stray characters) becomes NaN ("missing").
    column = pd.to_numeric(column, errors='coerce')

    # Remove physically impossible values (sensor glitches) using the
    # realistic range for THIS specific parameter.
    column[(column < min_valid) | (column > max_valid)] = np.nan

    return column


# Realistic (min, max) range for each parameter. These are generous
# bounds meant to catch clear sensor errors, not to filter normal
# variation -- adjust them if you know better limits for your station.
VALID_RANGES = {
    'SO2':          (0, 1000),   # ppb
    'NO':           (0, 1000),   # ppb
    'NO2':          (0, 1000),   # ppb
    'NOX':          (0, 2000),   # ppb
    'CO':           (0, 50),     # ppm
    'Co 8hr':       (0, 50),     # ppm
    'O3':           (0, 500),    # ppb
    'O38hr':        (0, 500),    # ppb
    'PM2.5':        (0, 1000),   # ug/m3
    'PM10':         (0, 1000),   # ug/m3
    'Wind Speed':   (0, 50),     # m/s
    'V Wind Speed': (0, 50),     # m/s
    'Wind Dir':     (0, 360),    # degrees (compass direction)
    'Temperature':  (-10, 50),   # deg C (realistic for Bangladesh)
    'RH':           (0, 100),    # % relative humidity
    'Solar Rad':    (0, 1500),   # W/m2
    'BP':           (850, 1100), # mb (barometric pressure)
    'Rain':         (0, 500),    # mm (hourly rainfall)
}

# Apply the cleaning to every column that exists in this sheet AND
# has a range defined above. (Some stations may be missing a column,
# e.g. no "V Wind Speed" -- this loop skips those automatically.)
for column_name, (low, high) in VALID_RANGES.items():
    if column_name in df.columns:
        df[column_name] = clean_number_column(df[column_name], low, high)

In [26]:
# ============================================================
# STEP 3: Make sure we have one proper "Date" column
# ============================================================
# Convert the Date column to an actual date/time type (not just text).
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Some sheets store just the date at midnight (with hour in a separate
# "Time" column); others already store the full date+time together.
# ".dt.normalize()" strips off any time-of-day and keeps just the
# calendar date - this makes both formats consistent before grouping.
df['CalendarDate'] = df['Date'].dt.normalize()


In [30]:
# ============================================================
# STEP 4: Group by day and calculate the daily average
# ============================================================
# groupby() collects all rows that share the same CalendarDate together.
# .agg() then tells pandas what to compute for each group:
#   - the MEAN (average) of PM2.5
#   - the MEAN (average) of PM10
#   - a COUNT of how many valid hourly PM2.5 readings went into that mean
#------------------------------------------------------------
# Daily aggregation
# ------------------------------------------------------------
daily = df.groupby('CalendarDate').agg(
    SO2_daily_avg=('SO2', 'mean'),
    NO_daily_avg=('NO', 'mean'),
    NO2_daily_avg=('NO2', 'mean'),
    NOX_daily_avg=('NOX', 'mean'),
    CO_daily_avg=('CO', 'mean'),
    CO8hr_daily_avg=('Co 8hr', 'mean'),
    O3_daily_avg=('O3', 'mean'),
    O38hr_daily_avg=('O38hr', 'mean'),
    PM25_daily_avg=('PM2.5', 'mean'),
    PM10_daily_avg=('PM10', 'mean'),
    WindSpeed_daily_avg=('Wind Speed', 'mean'),
    Temp_daily_avg=('Temperature', 'mean'),
    RH_daily_avg=('RH', 'mean'),
    SolarRad_daily_avg=('Solar Rad', 'mean'),
    BP_daily_avg=('BP', 'mean'),
    Rain_daily_total=('Rain', 'sum'),
    VWindSpeed_daily_avg=('V Wind Speed', 'mean'),

    hours_available=('PM2.5', 'count')
).reset_index()


# ------------------------------------------------------------
# Circular mean function for Wind Direction
# ------------------------------------------------------------
def circular_mean_deg(series):
    angles = pd.to_numeric(series, errors='coerce').dropna()

    if len(angles) == 0:
        return np.nan

    # Convert degrees to radians
    radians = np.deg2rad(angles)

    # Mean sine and cosine
    mean_sin = np.mean(np.sin(radians))
    mean_cos = np.mean(np.cos(radians))

    # Convert back to degrees
    mean_angle = np.rad2deg(np.arctan2(mean_sin, mean_cos))

    # Convert range from (-180, 180] to [0, 360)
    return mean_angle % 360

# ------------------------------------------------------------
# Calculate circular mean of Wind Direction separately
# ------------------------------------------------------------
wind_dir_daily = (
    df.groupby('CalendarDate')['Wind Dir']
    .apply(circular_mean_deg)
    .reset_index(name='WindDir_daily_circular_mean')
)


# ------------------------------------------------------------
# Merge Wind Direction with daily dataset
# ------------------------------------------------------------
daily = daily.merge(
    wind_dir_daily,
    on='CalendarDate',
    how='left'
)

print(f"\nGrouped into {len(daily)} calendar days.")


Grouped into 3235 calendar days.


In [31]:
# ============================================================
# STEP 5: Keep only "reliable" days
# ============================================================
# If a day only has, say, 2 hourly readings out of 24, its average
# isn't very trustworthy. We only keep days with enough data.
daily_reliable = daily[daily['hours_available'] >= MIN_HOURS_REQUIRED]

print(f"{len(daily_reliable)} days have at least {MIN_HOURS_REQUIRED} "
      f"hourly readings and are kept.")
print(f"{len(daily) - len(daily_reliable)} days had too few readings "
      f"and were dropped.")

2123 days have at least 18 hourly readings and are kept.
1112 days had too few readings and were dropped.


In [41]:
# ============================================================
# STEP 6: Save the result
# ============================================================

import os

# Output directory
OUTPUT_DIR = r"C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/CAMS_CleanDataset"

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Automatically create filename based on sheet name
OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{SHEET_NAME}_daily.csv"
)

# Save daily reliable data
daily_reliable.to_csv(OUTPUT_FILE, index=False)

print(f"\nDone! Daily data saved to: {OUTPUT_FILE}")
print("\nPreview of the result:")
print(daily_reliable.head())


Done! Daily data saved to: C:/Users/Juvair/Desktop/Job Application CV's/BUET_RA/Dataset/Air Pollution Dataset/CAMS_CleanDataset\BARC_daily1.csv

Preview of the result:
   CalendarDate  SO2_daily_avg  NO_daily_avg  NO2_daily_avg  NOX_daily_avg  \
1    2012-11-02       0.110556      7.334167       5.915833      13.249167   
2    2012-11-03       0.111818     10.431250       6.966667      17.398333   
3    2012-11-04       0.450000      9.399474       5.757059      15.483889   
4    2012-11-05       0.568636      9.777500       5.004545      15.029565   
11   2012-11-12       1.035000     13.445294       8.252500      20.495556   

    CO_daily_avg  CO8hr_daily_avg  O3_daily_avg  O38hr_daily_avg  \
1       0.094000         0.095005     24.831667        24.783958   
2       0.155217         0.149427     29.176818        31.937865   
3       0.127000         0.124461     30.668095        33.545992   
4       0.173846         0.157001     19.359524        19.351647   
11      5.802353      